In [96]:
import pandas as pd
import numpy as np
import sqlite3

customers = pd.read_csv("../data/olist_customers_dataset.csv")
orders = pd.read_csv("../data/olist_orders_dataset.csv")
order_items = pd.read_csv("../data/olist_order_items_dataset.csv")
payments = pd.read_csv("../data/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../data/olist_order_reviews_dataset.csv")
products = pd.read_csv("../data/olist_products_dataset.csv")
sellers = pd.read_csv("../data/olist_sellers_dataset.csv")
geolocation = pd.read_csv("../data/olist_geolocation_dataset.csv")
category_translation = pd.read_csv("../data/product_category_name_translation.csv")

print("All 9 datasets loaded")
print(f"Orders: {orders.shape}, Customers: {customers.shape}, Order Items: {order_items.shape}")

All 9 datasets loaded
Orders: (99441, 8), Customers: (99441, 5), Order Items: (112650, 7)


In [97]:
conn = sqlite3.connect("../data/olist.db")

customers.to_sql("customers", conn, if_exists="replace", index=False)
orders.to_sql("orders", conn, if_exists="replace", index=False)
order_items.to_sql("order_items", conn, if_exists="replace", index=False)
payments.to_sql("payments", conn, if_exists="replace", index=False)
reviews.to_sql("reviews", conn, if_exists="replace", index=False)
products.to_sql("products", conn, if_exists="replace", index=False)
sellers.to_sql("sellers", conn, if_exists="replace", index=False)
geolocation.to_sql("geolocation", conn, if_exists="replace", index=False)
category_translation.to_sql("category_translation", conn, if_exists="replace", index=False)

print("Database created with 9 tables")

Database created with 9 tables


In [98]:
query=""" 
select name from sqlite_master where type='table';
"""
tables = pd.read_sql_query(query, conn)
print(tables)

                   name
0             customers
1                orders
2           order_items
3              payments
4               reviews
5              products
6               sellers
7           geolocation
8  category_translation


In [99]:
query = """ 
SELECT 
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp,
    c.customer_state,
    p.payment_type,
    p.payment_value
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN payments p ON o.order_id = p.order_id
LIMIT 10;
"""
result = pd.read_sql_query(query, conn)
print(result)

                           order_id order_status order_purchase_timestamp  \
0  e481f51cbdc54678b7cc49136f2d6af7    delivered      2017-10-02 10:56:33   
1  e481f51cbdc54678b7cc49136f2d6af7    delivered      2017-10-02 10:56:33   
2  e481f51cbdc54678b7cc49136f2d6af7    delivered      2017-10-02 10:56:33   
3  53cdb2fc8bc7dce0b6741e2150273451    delivered      2018-07-24 20:41:37   
4  47770eb9100c2d0c44946d9cf07ec65d    delivered      2018-08-08 08:38:49   
5  949d5b44dbf5de918fe9c16f97b45f8a    delivered      2017-11-18 19:28:06   
6  ad21c59c0840e6cb83a9ceb5573f8159    delivered      2018-02-13 21:18:39   
7  a4591c265e18cb1dcee52889e2d8acc3    delivered      2017-07-09 21:57:05   
8  136cce7faa42fdb2cefd53fdc79a6098     invoiced      2017-04-11 12:22:08   
9  6514b8ad8028c9f2cc2374ded245783f    delivered      2017-05-16 13:10:30   

  customer_state payment_type  payment_value  
0             SP  credit_card          18.12  
1             SP      voucher           2.00  
2          

In [100]:
tables_info = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
    "category_translation": category_translation
}

for name, df in tables_info.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")
    print(f"  Columns: {list(df.columns)}")
    print()

customers: 99441 rows, 5 columns
  Columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

orders: 99441 rows, 8 columns
  Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

order_items: 112650 rows, 7 columns
  Columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

payments: 103886 rows, 5 columns
  Columns: ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

reviews: 99224 rows, 7 columns
  Columns: ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

products: 32951 rows, 9 columns
  Columns: ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product

In [101]:
query = """
SELECT 
    order_status,
    COUNT(*) as count,
    SUM(CASE WHEN order_delivered_customer_date IS NULL THEN 1 ELSE 0 END) as missing_delivered_date
FROM orders
GROUP BY order_status
ORDER BY count DESC;
"""
result = pd.read_sql_query(query, conn)
print(result)


  order_status  count  missing_delivered_date
0    delivered  96478                       8
1      shipped   1107                    1107
2     canceled    625                     619
3  unavailable    609                     609
4     invoiced    314                     314
5   processing    301                     301
6      created      5                       5
7     approved      2                       2


In [102]:
query = """ 
select 
    order_id,
    customer_id,
    order_purchase_timestamp,
    order_delivered_customer_date,
    order_estimated_delivery_date,
    julianday(order_delivered_customer_date) - julianday(order_estimated_delivery_date) as days_late
from orders
where order_status='delivered' and order_delivered_customer_date is not null
limit 10;
"""
result = pd.read_sql_query(query, conn)
print(result)

                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   
5  a4591c265e18cb1dcee52889e2d8acc3  503740e9ca751ccdda7ba28e9ab8f608   
6  6514b8ad8028c9f2cc2374ded245783f  9bdf08b4b3b52b5526ff42d37d47f222   
7  76c6e866289321a7c93b82b54852dc33  f54a9f0e6b351c431402b8461ea51999   
8  e69bfb5eb88e0ed6a785585b27e16dbf  31ad1d1b63eb9962463f764d4e6e0c9d   
9  e6ce16cb79ec1d90b1da9085a6118aeb  494dded5b201313c64ed7f100595b95c   

  order_purchase_timestamp order_delivered_customer_date  \
0      2017-10-02 10:56:33           2017-10-10 21:25:13   
1      2018-07-24 20:41:37           2018-08-07 15:27:45   
2      2018-08-0

In [103]:
query="""
select 
    case when julianday(order_delivered_customer_date) - julianday(order_estimated_delivery_date) > 0
         then 'Late' else 'On-time/Early' end as delivery_status,
    count(*) as count
from orders
where order_status='delivered' and order_delivered_customer_date is not null
Group by delivery_status;
"""
result = pd.read_sql_query(query, conn)
print(result)

  delivery_status  count
0            Late   7826
1   On-time/Early  88644


In [104]:
query="""
select 
    o.order_id,
    o.customer_id,
    o.order_purchase_timestamp,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    case when julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) > 0
         then 1 else 0 end as is_late
from orders o
where o.order_status='delivered' 
    and o.order_delivered_customer_date is not null
"""

base = pd.read_sql_query(query, conn)
print(base.shape)
print(base["is_late"].value_counts(normalize=True))

(96470, 6)
is_late
0    0.918876
1    0.081124
Name: proportion, dtype: float64


In [105]:
query=""" 
select 
    o.order_id,
    o.customer_id,
    CASE WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) > 0 
         THEN 1 ELSE 0 END as is_late,
    julianday(o.order_approved_at) - julianday(o.order_purchase_timestamp) as approval_days,
    julianday(o.order_delivered_carrier_date) - julianday(o.order_approved_at) as processing_days,
    julianday(o.order_delivered_customer_date) - julianday(o.order_delivered_carrier_date) as shipping_days,
    julianday(o.order_estimated_delivery_date) - julianday(o.order_purchase_timestamp) as estimated_delivery_days,
    strftime('%w', o.order_purchase_timestamp) as purchase_day_of_week,
    strftime('%m', o.order_purchase_timestamp) as purchase_month
from orders o
where o.order_status='delivered' and o.order_delivered_customer_date is not null;
"""

base = pd.read_sql_query(query, conn)
print(base.shape)
print(base.head())

(96470, 9)
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

   is_late  approval_days  processing_days  shipping_days  \
0        0       0.007431         2.366493       6.062650   
1        0       1.279745         0.462882      12.039410   
2        0       0.011505         0.204595       9.178113   
3        0       0.012419         3.745833       9.450498   
4        0       0.042940         0.893113       1.937824   

   estimated_delivery_days purchase_day_of_week purchase_month  
0                15.544062                    1             10  
1                19.137766                    2      

In [106]:
print(base.isnull().sum())


order_id                    0
customer_id                 0
is_late                     0
approval_days              14
processing_days            15
shipping_days               1
estimated_delivery_days     0
purchase_day_of_week        0
purchase_month              0
dtype: int64


In [107]:
print(base[base["approval_days"].isnull()][["order_id", "approval_days", "processing_days", "shipping_days"]])

                               order_id  approval_days  processing_days  \
5170   e04abd8149ef81b95221e88f6ed9ab6a            NaN              NaN   
16097  8a9adc69528e1001fc68dd0aaebbb54a            NaN              NaN   
18493  7013bcfc1c97fe719a7b5e05e61c12db            NaN              NaN   
21997  5cf925b116421afa85ee25e99b4c34fb            NaN              NaN   
22476  12a95a3c06dbaec84bcfb0e2da5d228a            NaN              NaN   
26012  c1d4211b3dae76144deccd6c74144a88            NaN              NaN   
37156  d69e5d356402adc8cf17e08b5033acfb            NaN              NaN   
38169  d77031d6a3c8a52f019764e68f211c69            NaN              NaN   
46954  7002a78c79c519ac54022d4f8a65e6e8            NaN              NaN   
59909  2eecb0d85f281280f79fa00f9cec1a95            NaN              NaN   
61159  51eb2eebd5d76a24625b31c33dd41449            NaN              NaN   
65675  88083e8f64d95b932164187484d90212            NaN              NaN   
70256  3c0b8706b065f9919d

In [108]:
base = base.dropna(subset=["approval_days", "processing_days", "shipping_days"])
print(base.shape)
print(base.isnull().sum().sum())

(96455, 9)
0


In [109]:
query = """
SELECT 
    order_id,
    COUNT(*) as num_items,
    SUM(price) as total_price,
    SUM(freight_value) as total_freight,
    AVG(price) as avg_item_price,
    COUNT(DISTINCT seller_id) as num_sellers
FROM order_items
GROUP BY order_id;
"""
items_agg = pd.read_sql_query(query, conn)
print(items_agg.shape)
print(items_agg.head())

(98666, 6)
                           order_id  num_items  total_price  total_freight  \
0  00010242fe8c5a6d1ba2dd792cb16214          1        58.90          13.29   
1  00018f77f2f0320c557190d7a144bdd3          1       239.90          19.93   
2  000229ec398224ef6ca0657da4fc703e          1       199.00          17.87   
3  00024acbcdf0a6daa1e931b038114c75          1        12.99          12.79   
4  00042b26cf59d7ce69dfabb4e55b4fd9          1       199.90          18.14   

   avg_item_price  num_sellers  
0           58.90            1  
1          239.90            1  
2          199.00            1  
3           12.99            1  
4          199.90            1  


In [110]:
base = base.merge(items_agg, on="order_id", how="left")
print(base.shape)
print(base.isnull().sum())


(96455, 14)
order_id                   0
customer_id                0
is_late                    0
approval_days              0
processing_days            0
shipping_days              0
estimated_delivery_days    0
purchase_day_of_week       0
purchase_month             0
num_items                  0
total_price                0
total_freight              0
avg_item_price             0
num_sellers                0
dtype: int64


In [111]:
query = """
SELECT 
    order_id,
    SUM(payment_value) as total_payment_value,
    AVG(payment_installments) as avg_installments,
    COUNT(DISTINCT payment_type) as num_payment_types
FROM payments
GROUP BY order_id;
"""
payments_agg = pd.read_sql_query(query, conn)

base = base.merge(payments_agg, on="order_id", how="left")
print(base.shape)
print(base.isnull().sum())

(96455, 17)
order_id                   0
customer_id                0
is_late                    0
approval_days              0
processing_days            0
shipping_days              0
estimated_delivery_days    0
purchase_day_of_week       0
purchase_month             0
num_items                  0
total_price                0
total_freight              0
avg_item_price             0
num_sellers                0
total_payment_value        1
avg_installments           1
num_payment_types          1
dtype: int64


In [112]:
print(base[base["total_payment_value"].isnull()])

                               order_id                       customer_id  \
29803  bfbd0f9bdef84302105ad712db648a6c  86dc2ffce2dfff336de2f386a786e574   

       is_late  approval_days  processing_days  shipping_days  \
29803        1            0.0        53.205035        1.60816   

       estimated_delivery_days purchase_day_of_week purchase_month  num_items  \
29803                18.488449                    4             09          3   

       total_price  total_freight  avg_item_price  num_sellers  \
29803       134.97           8.49           44.99            1   

       total_payment_value  avg_installments  num_payment_types  
29803                  NaN               NaN                NaN  


In [113]:
base = base.dropna(subset=["total_payment_value", "avg_installments", "num_payment_types"])
print(base.shape)
print(base.isnull().sum().sum())

(96454, 17)
0


In [114]:
query="""
select
    oi.order_id,
    avg(p.product_weight_g) as avg_weight_g,
    avg(p.product_length_cm * p.product_height_cm * p.product_width_cm) as avg_volume_cm3,
    count(distinct p.product_category_name) as num_categories
from order_items oi
join products p on oi.product_id = p.product_id
group by oi.order_id;
"""
products_agg = pd.read_sql_query(query, conn)
base = base.merge(products_agg, on="order_id", how="left")
print(base.shape)
print(base.isnull().sum())

(96454, 20)
order_id                    0
customer_id                 0
is_late                     0
approval_days               0
processing_days             0
shipping_days               0
estimated_delivery_days     0
purchase_day_of_week        0
purchase_month              0
num_items                   0
total_price                 0
total_freight               0
avg_item_price              0
num_sellers                 0
total_payment_value         0
avg_installments            0
num_payment_types           0
avg_weight_g               16
avg_volume_cm3             16
num_categories              0
dtype: int64


In [115]:
base = base.dropna(subset=["avg_weight_g", "avg_volume_cm3"])
print(base.shape)
print(base.isnull().sum().sum())

(96438, 20)
0


In [116]:
query = """
SELECT 
    geolocation_zip_code_prefix as zip_code,
    AVG(geolocation_lat) as lat,
    AVG(geolocation_lng) as lng
FROM geolocation
GROUP BY geolocation_zip_code_prefix;
"""
geo_agg = pd.read_sql_query(query, conn)
print(geo_agg.shape)
print(geo_agg.head())

(19015, 3)
   zip_code        lat        lng
0      1001 -23.550190 -46.634024
1      1002 -23.548146 -46.634979
2      1003 -23.548994 -46.635731
3      1004 -23.549799 -46.634757
4      1005 -23.549456 -46.636733


In [117]:
query = """
SELECT DISTINCT
    o.order_id,
    c.customer_zip_code_prefix as customer_zip
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id;
"""
customer_locations = pd.read_sql_query(query, conn)

query = """
SELECT 
    oi.order_id,
    s.seller_zip_code_prefix as seller_zip
FROM order_items oi
JOIN sellers s ON oi.seller_id = s.seller_id
GROUP BY oi.order_id;
"""
seller_locations = pd.read_sql_query(query, conn)

print(customer_locations.shape)
print(seller_locations.shape)


(99441, 2)
(98666, 2)


In [118]:
customer_locations = customer_locations.merge(
    geo_agg, left_on="customer_zip", right_on="zip_code", how="left"
).rename(columns={"lat": "customer_lat", "lng": "customer_lng"})

seller_locations = seller_locations.merge(
    geo_agg, left_on="seller_zip", right_on="zip_code", how="left"
).rename(columns={"lat": "seller_lat", "lng": "seller_lng"})

locations = customer_locations.merge(seller_locations, on="order_id", how="inner")

locations["distance_km"] = np.sqrt(
    (locations["customer_lat"] - locations["seller_lat"])**2 +
    (locations["customer_lng"] - locations["seller_lng"])**2
) * 111

print(locations.shape)
print(locations[["order_id", "customer_lat", "seller_lat", "distance_km"]].head())

(98666, 10)
                           order_id  customer_lat  seller_lat  distance_km
0  e481f51cbdc54678b7cc49136f2d6af7    -23.576983  -23.680729    19.603456
1  53cdb2fc8bc7dce0b6741e2150273451    -12.177924  -19.807681   850.262665
2  47770eb9100c2d0c44946d9cf07ec65d    -16.745150  -21.363502   513.613515
3  949d5b44dbf5de918fe9c16f97b45f8a     -5.774190  -19.837682  1832.860509
4  ad21c59c0840e6cb83a9ceb5573f8159    -23.676370  -23.543395    31.680666


In [119]:
base = base.merge(locations[["order_id", "distance_km"]], on="order_id", how="left")
print(base.shape)
print(base.isnull().sum())

(96438, 21)
order_id                     0
customer_id                  0
is_late                      0
approval_days                0
processing_days              0
shipping_days                0
estimated_delivery_days      0
purchase_day_of_week         0
purchase_month               0
num_items                    0
total_price                  0
total_freight                0
avg_item_price               0
num_sellers                  0
total_payment_value          0
avg_installments             0
num_payment_types            0
avg_weight_g                 0
avg_volume_cm3               0
num_categories               0
distance_km                480
dtype: int64


In [120]:
missing_distance = base[base["distance_km"].isnull()]
print(missing_distance.shape)

# Check if it's a customer zip issue, seller zip issue, or both
check = locations[locations["order_id"].isin(missing_distance["order_id"])]
print(check[["customer_lat", "seller_lat"]].isnull().sum())

(480, 21)
customer_lat    265
seller_lat      216
dtype: int64


In [121]:
base = base.dropna(subset=["distance_km"])
print(base.shape)
print(base.isnull().sum().sum())

(95958, 21)
0


In [122]:
query = """
SELECT 
    oi.order_id,
    ct.product_category_name_english as category
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
LEFT JOIN category_translation ct ON p.product_category_name = ct.product_category_name
GROUP BY oi.order_id;
"""
category_data = pd.read_sql_query(query, conn)

base = base.merge(category_data, on="order_id", how="left")
print(base.shape)
print(base.isnull().sum())
print(base["category"].nunique())

(95958, 22)
order_id                      0
customer_id                   0
is_late                       0
approval_days                 0
processing_days               0
shipping_days                 0
estimated_delivery_days       0
purchase_day_of_week          0
purchase_month                0
num_items                     0
total_price                   0
total_freight                 0
avg_item_price                0
num_sellers                   0
total_payment_value           0
avg_installments              0
num_payment_types             0
avg_weight_g                  0
avg_volume_cm3                0
num_categories                0
distance_km                   0
category                   1356
dtype: int64
71


In [123]:
missing_cat = base[base["category"].isnull()]
print(missing_cat.shape)
print(missing_cat["is_late"].value_counts(normalize=True))


(1356, 22)
is_late
0    0.909292
1    0.090708
Name: proportion, dtype: float64


In [124]:
base["category"] = base["category"].fillna("Unknown")
print(base.isnull().sum().sum())
print(base.shape)


0
(95958, 22)


In [125]:
print(base.dtypes)
print()
print(base.shape)
base.head()

order_id                       str
customer_id                    str
is_late                      int64
approval_days              float64
processing_days            float64
shipping_days              float64
estimated_delivery_days    float64
purchase_day_of_week           str
purchase_month                 str
num_items                    int64
total_price                float64
total_freight              float64
avg_item_price             float64
num_sellers                  int64
total_payment_value        float64
avg_installments           float64
num_payment_types          float64
avg_weight_g               float64
avg_volume_cm3             float64
num_categories               int64
distance_km                float64
category                       str
dtype: object

(95958, 22)


,order_id,customer_id,is_late,approval_days,processing_days,shipping_days,estimated_delivery_days,purchase_day_of_week,purchase_month,num_items,...,avg_item_price,num_sellers,total_payment_value,avg_installments,num_payment_types,avg_weight_g,avg_volume_cm3,num_categories,distance_km,category
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,0,0.007431,2.366493,6.062650,15.544062,1,10,1,...,29.99,1,38.71,1.0,2.0,500.0,1976.0,1,19.603456,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,0,1.279745,0.462882,12.039410,19.137766,2,07,1,...,118.70,1,141.46,1.0,1.0,400.0,4693.0,1,850.262665,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,0,0.011505,0.204595,9.178113,26.639711,3,08,1,...,159.90,1,179.12,3.0,1.0,420.0,9576.0,1,513.613515,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,0,0.012419,3.745833,9.450498,26.188819,6,11,1,...,45.00,1,72.20,1.0,1.0,450.0,6000.0,1,1832.860509,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,0,0.042940,0.893113,1.937824,12.112049,2,02,1,...,19.90,1,28.62,1.0,1.0,250.0,11475.0,1,31.680666,stationery


In [126]:
model_df = base.drop(columns=["order_id", "customer_id"])

model_df["purchase_day_of_week"] = model_df["purchase_day_of_week"].astype(int)
model_df["purchase_month"] = model_df["purchase_month"].astype(int)

model_df = pd.get_dummies(model_df, columns=["category"], drop_first=True)

print(model_df.shape)
print(model_df.dtypes.value_counts())

(95958, 90)
bool       71
float64    13
int64       6
Name: count, dtype: int64


In [127]:
from sklearn.model_selection import train_test_split

X = model_df.drop(columns=["is_late"])
y = model_df["is_late"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(76766, 89) (19192, 89)
is_late
0    0.918988
1    0.081012
Name: proportion, dtype: float64
is_late
0    0.918977
1    0.081023
Name: proportion, dtype: float64


In [128]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
log_reg.fit(X_train_scaled, y_train)

log_reg_preds = log_reg.predict(X_test_scaled)

print("Test accuracy:", log_reg.score(X_test_scaled, y_test))
print(confusion_matrix(y_test, log_reg_preds))
print(classification_report(y_test, log_reg_preds))

Test accuracy: 0.9885889954147562
[[17418   219]
 [    0  1555]]
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     17637
           1       0.88      1.00      0.93      1555

    accuracy                           0.99     19192
   macro avg       0.94      0.99      0.96     19192
weighted avg       0.99      0.99      0.99     19192



In [129]:
model_df_clean = model_df.drop(columns=["shipping_days", "processing_days", "approval_days"])
print(model_df_clean.shape)
print(model_df_clean.columns.tolist())

(95958, 87)
['is_late', 'estimated_delivery_days', 'purchase_day_of_week', 'purchase_month', 'num_items', 'total_price', 'total_freight', 'avg_item_price', 'num_sellers', 'total_payment_value', 'avg_installments', 'num_payment_types', 'avg_weight_g', 'avg_volume_cm3', 'num_categories', 'distance_km', 'category_agro_industry_and_commerce', 'category_air_conditioning', 'category_art', 'category_arts_and_craftmanship', 'category_audio', 'category_auto', 'category_baby', 'category_bed_bath_table', 'category_books_general_interest', 'category_books_imported', 'category_books_technical', 'category_cds_dvds_musicals', 'category_christmas_supplies', 'category_cine_photo', 'category_computers', 'category_computers_accessories', 'category_consoles_games', 'category_construction_tools_construction', 'category_construction_tools_lights', 'category_construction_tools_safety', 'category_cool_stuff', 'category_costruction_tools_garden', 'category_costruction_tools_tools', 'category_diapers_and_hygien

In [130]:
X = model_df_clean.drop(columns=["is_late"])
y = model_df_clean["is_late"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

(76766, 86) (19192, 86)


In [131]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
log_reg.fit(X_train_scaled, y_train)

log_reg_preds = log_reg.predict(X_test_scaled)

print("Test accuracy:", log_reg.score(X_test_scaled, y_test))
print(confusion_matrix(y_test, log_reg_preds))
print(classification_report(y_test, log_reg_preds))

Test accuracy: 0.6057211338057524
[[10675  6962]
 [  605   950]]
              precision    recall  f1-score   support

           0       0.95      0.61      0.74     17637
           1       0.12      0.61      0.20      1555

    accuracy                           0.61     19192
   macro avg       0.53      0.61      0.47     19192
weighted avg       0.88      0.61      0.69     19192



In [132]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, class_weight="balanced")
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

print("--- Random Forest ---")
print("Test accuracy:", rf.score(X_test, y_test))
print(confusion_matrix(y_test, rf_preds))
print(classification_report(y_test, rf_preds))

gb = GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=42)
gb.fit(X_train, y_train)
gb_preds = gb.predict(X_test)

print("--- Gradient Boosting ---")
print("Test accuracy:", gb.score(X_test, y_test))
print(confusion_matrix(y_test, gb_preds))
print(classification_report(y_test, gb_preds))

--- Random Forest ---
Test accuracy: 0.7446331804918717
[[13363  4274]
 [  627   928]]
              precision    recall  f1-score   support

           0       0.96      0.76      0.85     17637
           1       0.18      0.60      0.27      1555

    accuracy                           0.74     19192
   macro avg       0.57      0.68      0.56     19192
weighted avg       0.89      0.74      0.80     19192

--- Gradient Boosting ---
Test accuracy: 0.919237182159233
[[17609    28]
 [ 1522    33]]
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     17637
           1       0.54      0.02      0.04      1555

    accuracy                           0.92     19192
   macro avg       0.73      0.51      0.50     19192
weighted avg       0.89      0.92      0.88     19192



In [133]:
from sklearn.model_selection import cross_val_score

models_cv = {
    "Logistic Regression": (log_reg, X_train_scaled),
    "Random Forest": (rf, X_train),
    "Gradient Boosting": (gb, X_train)
}

for name, (model, X_cv) in models_cv.items():
    scores = cross_val_score(model, X_cv, y_train, cv=5, scoring="f1")
    print(f"{name} - CV F1 (late class): {scores.mean():.4f} ± {scores.std():.4f}")

Logistic Regression - CV F1 (late class): 0.2000 ± 0.0041
Random Forest - CV F1 (late class): 0.2701 ± 0.0050
Gradient Boosting - CV F1 (late class): 0.0264 ± 0.0048


In [134]:
importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importances.head(15))

purchase_month             0.269095
estimated_delivery_days    0.223794
distance_km                0.129632
total_freight              0.081594
total_payment_value        0.038630
avg_weight_g               0.036506
avg_item_price             0.035172
avg_volume_cm3             0.034655
total_price                0.032227
num_sellers                0.016192
purchase_day_of_week       0.014856
avg_installments           0.013818
num_items                  0.008665
category_bed_bath_table    0.005094
num_categories             0.004935
dtype: float64


In [135]:
sample = X_test.iloc[:10]
actual = y_test.iloc[:10]
predicted = rf.predict(sample)
probabilities = rf.predict_proba(sample)[:, 1]

comparison = pd.DataFrame({
    "Actual": actual.values,
    "Predicted": predicted,
    "Late Probability": probabilities.round(3),
    "Correct": actual.values == predicted
})

print(comparison)

   Actual  Predicted  Late Probability  Correct
0       0          0             0.410     True
1       0          1             0.542    False
2       0          1             0.542    False
3       0          1             0.539    False
4       0          0             0.277     True
5       0          0             0.359     True
6       1          0             0.467    False
7       0          0             0.415     True
8       0          0             0.411     True
9       0          0             0.498     True
